# S2 — Pandas I/O 與資料清理

> **時間**：2 小時  
> **資料**：`datasets/ecommerce/orders_raw.csv`（故意弄髒的訂單檔）  
> **學完能幹嘛**：拿到一份髒資料，能在 15 分鐘內清成乾淨的 DataFrame

---

## 上節回顧 + 為什麼要學 Pandas

S1 我們學了 NumPy，但它只能處理**純數字矩陣**。真實世界的資料長這樣：

```
Order_ID , customer_id, Product_ID,  qty, order_date, amount
5064    , 2022       , 1026      ,    4, 2025-03-26, 8600
5023    , 2026       , 1021      ,    5,          , "$1,355"
```

問題一大堆：
- 欄名**有空白**（`Order_ID ` 尾端一個空格）
- `amount` 有時是 `8600`、有時是 `"$1,355"`
- `order_date` 有**缺值**
- 還有**重複列**

**Pandas = NumPy + 欄名 + 索引 + 缺值處理 + 各種資料型別**，專治這種髒資料。

> **DE 視角：資料清理 (Data Cleaning) 是資料工程的第一哩路，通常佔專案 60~80% 時間。**


---
## 核心觀念 1／4 — DataFrame 與 Series

- **Series**：一維、有索引的陣列（像一個有標籤的 column）
- **DataFrame**：二維表格（像 Excel 的一張 sheet），每一個 column 就是一個 Series


In [ ]:
import pandas as pd
import numpy as np

# 從 dict 建 DataFrame
df = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Carol'],
    'age':   [25, 30, 28],
    'score': [85.5, 92.0, 78.3],
})

print(df)
print('\nshape   =', df.shape)   # (3, 3)
print('columns =', list(df.columns))
print('dtypes  =\n', df.dtypes)

# 取單一 column → Series
print('\ndf["age"] (Series):')
print(df['age'])

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from common.checker import check

---
## 核心觀念 2／4 — 讀檔：`read_csv` / `read_excel`

資料最常見的來源是 CSV 或 Excel。一行就能讀進來。


In [ ]:
DATA = '../datasets/ecommerce/orders_raw.csv'

raw = pd.read_csv(DATA)

print('資料形狀 =', raw.shape)
print('\n前 5 列:')
print(raw.head())

print('\n每欄型別:')
print(raw.dtypes)

print('\n每欄缺值數:')
print(raw.isna().sum())

**常用探索指令**（拿到任何新資料第一件事做這五行）：

| 指令 | 作用 |
|---|---|
| `df.head()` | 看前 5 列 |
| `df.tail()` | 看最後 5 列 |
| `df.shape` | 幾列幾欄 |
| `df.dtypes` | 每欄型別 |
| `df.info()` | 綜合資訊 |
| `df.describe()` | 數值欄的統計摘要 |
| `df.isna().sum()` | 每欄缺值數 |


> ### 💡 知識補給站 — 大檔案不要一次讀完：chunksize
> 
> `pd.read_csv()` 預設把整份檔案讀進記憶體。如果 CSV 有 500 萬列，記憶體直接爆掉。
> 
> 加上 `chunksize=10000` 後，`read_csv` 會回傳一個 **iterator**，每次只給你 1 萬列：
> 
> ```python
> total = 0
> for chunk in pd.read_csv('big_file.csv', chunksize=10000):
>     total += chunk['amount'].sum()
> ```
> 
> 類比：吃到飽餐廳不會一次把所有菜端上桌，你一盤一盤拿 → 這就是 chunked processing。
> 
> 口訣：「**小資料全讀，中資料分批，大資料換引擎**」（Dask / Polars）。
> 
> *延伸關鍵字：chunksize, iterator, chunked processing, memory efficiency*

---
## 核心觀念 3／4 — 選取資料：`loc` vs `iloc`

小白只需要記兩個：

- `df['欄名']` → 取一個 column
- `df.loc[條件]` → 用條件篩選列


In [ ]:
# 取單欄
print(raw['customer_id'].head())

# 取多欄（注意：兩層中括號）
print('\n')
print(raw[['customer_id', 'Product_ID']].head())

# 條件篩選：只看 customer_id == 2020 的訂單
mask = raw['customer_id'] == 2020
print('\n該顧客訂單:')
print(raw.loc[mask].head())

---
## 核心觀念 4／4 — 資料清理四大手法

1. **欄名標準化**：去空白、轉小寫
2. **型別轉換**：字串金額 → 數字、字串日期 → datetime
3. **缺值處理**：`dropna` 或 `fillna`
4. **重複列處理**：`drop_duplicates`


---
## 實務 Case — 把 `orders_raw.csv` 清乾淨 🧹

我們一步步把上面的髒資料清成**可以直接分析**的狀態。


In [ ]:
# Step 0：再看一次原始檔
df = pd.read_csv('../datasets/ecommerce/orders_raw.csv')
print('原始欄名:', list(df.columns))
print('原始形狀:', df.shape)
df.head(3)

In [ ]:
# Step 1：欄名標準化 —— 去空白、全部轉小寫
df.columns = df.columns.str.strip().str.lower()
print('清理後欄名:', list(df.columns))

In [ ]:
# Step 2：把 amount 從 '$1,355' 這種字串轉成數字
#   技巧：先把 $ 和 , 拿掉，再 astype(float)
print('清理前 amount 型別:', df['amount'].dtype)
print('範例亂值:', df['amount'].astype(str).str.contains('\\$').sum(), '個含 $ 符號')

df['amount'] = (
    df['amount']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)
print('\n清理後 amount 型別:', df['amount'].dtype)
print('amount 摘要:\n', df['amount'].describe())

In [ ]:
# Step 3：把 order_date 從字串轉 datetime，缺值會自動變 NaT
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
print('order_date 型別:', df['order_date'].dtype)
print('缺日期筆數:', df['order_date'].isna().sum())

In [ ]:
# Step 4：缺值處理策略
# - 沒有日期的訂單沒辦法做時序分析 → 丟掉
# - 沒有 qty 的訂單 → 用中位數補
print('清理前:', df.shape)

df = df.dropna(subset=['order_date'])
df['qty'] = df['qty'].fillna(df['qty'].median())

print('清理後:', df.shape)
print('剩餘缺值:\n', df.isna().sum())

> ### 💡 知識補給站 — 缺值不只一種：MCAR / MAR / MNAR
> 
> 我們剛才 `dropna` 和 `fillna(median)` 看似簡單，但缺值有**三種成因**，處理策略完全不同：
> 
> | 類型 | 全名 | 意義 | 處理 |
> |---|---|---|---|
> | **MCAR** | Missing Completely at Random | 純隨機遺漏（如系統偶爾斷線） | `dropna` 通常安全 |
> | **MAR** | Missing at Random | 遺漏跟**其他欄位**有關（如高消費客戶懶得填問卷） | 可用其他欄位推估補值 |
> | **MNAR** | Missing Not at Random | 遺漏跟**缺值本身**有關（如低薪員工不填薪資） | 最棘手，簡單 fillna 會引入偏差 (bias) |
> 
> 我們剛才 `dropna(subset=['order_date'])` 是假設日期缺失屬於 MCAR；`fillna(median)` 是假設 qty 缺失不影響分析結論。
> 
> 真實工作中，先問一句：「**這些值為什麼不見了？**」再決定策略。
> 
> *延伸關鍵字：MCAR, MAR, MNAR, imputation bias, missing data mechanism*

In [ ]:
# Step 5：重複列處理
print('重複列數:', df.duplicated().sum())
df = df.drop_duplicates()
print('去重後形狀:', df.shape)

In [ ]:
# Step 6：驗收清理成果
print('✅ 最終形狀:', df.shape)
print('✅ 欄名:', list(df.columns))
print('✅ 型別:')
print(df.dtypes)
print('\n✅ 摘要:')
df.describe(include='all').head()

> ### 💡 知識補給站 — Schema Validation：讓資料自動驗收
> 
> 我們剛才的「驗收」是人工看 `dtypes` 和 `isna().sum()`，但如果每天自動跑清理 pipeline，沒人盯怎麼辦？
> 
> **Schema Validation** = 自動檢查規則：每欄型別對不對？值的範圍合理嗎？有沒有不該出現的 null？
> 
> - 概念就像寫「**測試**」給你的資料 — 資料不符合規則就自動報錯
> - 業界工具：`pandera`、`great_expectations` — 一行程式就能定義「amount 必須 > 0」「order_date 不能有 null」
> - 類比：餐廳的食安檢查表 → 溫度、保存期限、清潔度，每項打勾才能出餐
> 
> *延伸關鍵字：schema validation, pandera, great_expectations, data contract*

In [ ]:
# Step 7：存檔給下一堂 S3 用
df.to_csv('../datasets/ecommerce/orders_clean.csv', index=False)
print('已存檔 → orders_clean.csv（S3 會直接讀這份）')

🎉 **恭喜！你剛剛完成了一次真實的 Data Engineering 小任務**：Extract（讀檔）→ Transform（清理）→ Load（存檔）= ETL 的縮影。


---
## 課堂練習（40 min）

### 🟢 送分題
建立下面這個 DataFrame，然後：
1. 用 `.head(2)` 看前兩列
2. 用 `.dtypes` 查看型別
3. 取出 `price` 這個 column


In [ ]:
practice = pd.DataFrame({
    'item':  ['Apple', 'Banana', 'Cherry', 'Durian'],
    'price': [50, 20, 150, 300],
    'stock': [10, 50, 5, 2],
})
# TODO: 你的程式碼


### 🟡 核心題
重新讀一次 `orders_raw.csv`，完成這些步驟並**回答問題**：
1. 清理後總共有多少筆訂單？
2. `amount` 的平均值是多少？
3. 請列出 amount > 5000 的訂單筆數


In [ ]:
# TODO: 你的程式碼


### 🔴 挑戰題
清理邏輯封裝成一個函式 `clean_orders(path) -> pd.DataFrame`，要求：
- 輸入：髒檔案路徑
- 輸出：清理完成的 DataFrame
- 至少處理：欄名空白、金額字串、日期、缺值、重複列
- 印出 1 行清理摘要（例如：`清理完成：原 210 筆 → 192 筆`）

**為什麼要封裝？** 未來你遇到新的訂單檔，只要呼叫一次函式就清好了 → 這就是 **DE 的可重用 pipeline**。


In [ ]:
def clean_orders(path):
    # TODO: 你的實作
    pass

# clean_orders('../datasets/ecommerce/orders_raw.csv')


In [ ]:
# 🏁 大地遊戲 — 清理完有幾筆訂單？答對就能找到解答！
check('S2', len(df))

---
## 小結與速查表

| 想做什麼 | 怎麼寫 |
|---|---|
| 讀 CSV | `pd.read_csv(path)` |
| 讀 Excel | `pd.read_excel(path, sheet_name=...)` |
| 快速瞄一眼 | `df.head() / df.info() / df.describe()` |
| 看缺值 | `df.isna().sum()` |
| 欄名去空白 | `df.columns = df.columns.str.strip()` |
| 字串 → 數字 | `df['x'].str.replace('$','').astype(float)` |
| 字串 → 日期 | `pd.to_datetime(df['x'], errors='coerce')` |
| 丟缺值 | `df.dropna(subset=['col'])` |
| 補缺值 | `df['x'].fillna(值)` |
| 去重 | `df.drop_duplicates()` |
| 存檔 | `df.to_csv(path, index=False)` |

**ETL 心法**：拿到任何新資料 → `head/info/isna` 快速掃一遍 → 逐欄處理 → 驗收 → 存乾淨檔給下游。

**下節預告 S3**：資料清乾淨了，但單一張表看不出商業洞察。我們會把 `orders_clean.csv` 和 `customers.csv`、`products.csv` **三表 join**，算出「各地區 Top 熱銷商品」「VIP 顧客貢獻佔比」。
